# Bài học 02 - Khám phá Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** là một framework thống nhất để xây dựng các tác nhân AI. Nó cung cấp một kiến trúc sạch sẽ, có thể cấu thành với bốn thành phần cốt lõi:

- **Client** – kết nối với điểm cuối mô hình AI và xử lý giao tiếp
- **Agent** – bao bọc client với các hướng dẫn và định nghĩa công cụ
- **Tools** – mở rộng khả năng của agent với các chức năng tùy chỉnh mà mô hình có thể gọi
- **Session** – duy trì lịch sử hội thoại cho các tương tác đa lượt

Trong bài học này, chúng ta sẽ xây dựng một **tác nhân đặt vé du lịch** kiểm tra tính khả dụng của điểm đến sử dụng các khái niệm này.


## Cài đặt


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Hiểu về Kiến trúc Khung Agent

Khung Agent của Microsoft tuân theo kiến trúc phân lớp:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Một `FoundryChatClient` kết nối với Azure OpenAI deployment. Nó xử lý việc xác thực, định dạng yêu cầu và phân tích phản hồi.
2. **Agent** – Được tạo từ client thông qua `provider.create_agent()`, agent kết hợp việc truy cập mô hình với các hướng dẫn (prompt hệ thống) và các công cụ.
3. **Tools** – Các hàm Python được trang trí bằng `@tool` mà agent có thể sử dụng để thực hiện hành động hoặc truy xuất dữ liệu.
4. **Session** – Một đối tượng `AgentSession` (được tạo thông qua `agent.create_session()`) lưu trữ lịch sử hội thoại, cho phép đối thoại nhiều lượt trong đó agent ghi nhớ ngữ cảnh trước đó.

Hãy cùng xây dựng từng lớp từng bước một.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Thêm Công cụ với Decorator @tool

Công cụ cho phép tác nhân thực hiện các hành động ngoài việc tạo văn bản. Decorator `@tool` chuyển một hàm Python thông thường thành thứ mà tác nhân có thể gọi.

Các điểm chính:
- Sử dụng `Annotated[type, "description"]` để mô hình hiểu mỗi tham số.
- Docstring trở thành mô tả công cụ mà mô hình thấy.
- `approval_mode="never_require"` nghĩa là công cụ chạy tự động mà không cần xác nhận của người dùng.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tạo một Tác nhân với Công cụ

Bây giờ chúng ta kết hợp client, hướng dẫn, và công cụ thành một tác nhân. `instructions` đóng vai trò như lời nhắc hệ thống — chúng định nghĩa cá tính và hành vi của tác nhân.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Các Cuộc Hội Thoại Nhiều Lượt với Phiên

Một `AgentSession` (được tạo qua `agent.create_session()`) theo dõi tất cả các tin nhắn trong một cuộc hội thoại. Bằng cách truyền cùng phiên cho mỗi lần gọi `agent.run()`, đại lý có thể truy cập toàn bộ lịch sử hội thoại và có thể tham khảo lại các tin nhắn trước đó.

Chúng tôi truyền `tools=[check_destination_availability]` để đại lý có thể gọi trình kiểm tra tính sẵn có của chúng tôi trong mỗi lượt.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Tóm tắt

Trong bài học này, bạn đã khám phá bốn trụ cột của Microsoft Agent Framework:

| Khái niệm | Những gì bạn đã học |
|---------|------------------|
| **Khách hàng** | `FoundryChatClient` kết nối với Azure OpenAI bằng xác thực dựa trên thông tin đăng nhập |
| **Đại lý** | `provider.create_agent()` kết hợp kết nối mô hình với hướng dẫn và tên |
| **Công cụ** | Trình trang trí `@tool` cho phép gọi các hàm Python cho đại lý |
| **Phiên** | `agent.create_session()` duy trì lịch sử hội thoại qua nhiều lượt |

Những khối xây dựng này kết hợp với nhau để tạo ra các đại lý có thể duy trì cuộc trò chuyện tự nhiên, gọi các hàm bên ngoài và giữ ngữ cảnh — nền tảng cho các mẫu đại lý nâng cao hơn trong các bài học sau.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
